### Dataset

In [1]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from transformers import TrOCRProcessor
from torch.utils.data import random_split, DataLoader
import torch

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")

class RxHandBDDataset(Dataset):
    def __init__(self, csv_path, image_dir, processor):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]  # Get the row at index idx

        image_path = os.path.join(self.image_dir, row["Images"])
        image = Image.open(image_path).convert("RGB")

        text = row["Text"]

        # Converting our image pixels to tensor
        pixel_values = self.processor(
            images=image,
            return_tensors="pt"
        ).pixel_values.squeeze(0)

        # Convert our ground truth to token ids
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=64,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze(0)

        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        # Dataloader automatically batches these so its fine
        return {
            "pixel_values": pixel_values,
            "labels": labels,
        }

In [2]:
dataset = RxHandBDDataset(
    "/workspace/rxhandbd/RxHandBD-ML/Train_Label.csv",
    "/workspace/rxhandbd/RxHandBD-ML/Train_Set",
    processor
)

val_fraction = 0.08

num_total = len(dataset)
num_val = int(num_total * val_fraction)
num_train = num_total - num_val

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    dataset,
    [num_train, num_val],
    generator=generator
)

print("train:", len(train_dataset))
print("val:", len(val_dataset))


train: 4106
val: 357


### DataLoader

In [3]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False
)

### Align the Model to the Processor. 
Make sure that start, pad, and eos are the same

In [4]:
def align_model_to_processor(model, processor):
    tokenizer = processor.tokenizer

    model.config.decoder_start_token_id = tokenizer.cls_token_id
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.sep_token_id

    model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.sep_token_id

    # Useful for generation later
    model.generation_config.max_length = 64
    model.generation_config.num_beams = 1

    return model

### Define the VisionEncoderDecoder Model

In [5]:
import torch
from transformers import VisionEncoderDecoderModel

device = torch.device("cuda")

model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-handwritten"
).to(device)

model = align_model_to_processor(model, processor)



Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Sanity check special token alignment

In [6]:
def check_trocr_alignment(model, processor):
    tokenizer = processor.tokenizer

    expected_decoder_start = tokenizer.cls_token_id
    expected_pad = tokenizer.pad_token_id
    expected_eos = tokenizer.sep_token_id

    checks = {
        "model.config.decoder_start_token_id": (
            model.config.decoder_start_token_id,
            expected_decoder_start,
        ),
        "model.config.pad_token_id": (
            model.config.pad_token_id,
            expected_pad,
        ),
        "model.config.eos_token_id": (
            model.config.eos_token_id,
            expected_eos,
        ),
        "model.generation_config.decoder_start_token_id": (
            model.generation_config.decoder_start_token_id,
            expected_decoder_start,
        ),
        "model.generation_config.pad_token_id": (
            model.generation_config.pad_token_id,
            expected_pad,
        ),
        "model.generation_config.eos_token_id": (
            model.generation_config.eos_token_id,
            expected_eos,
        ),
    }

    for name, (actual, expected) in checks.items():
        assert actual == expected, f"{name}: expected {expected}, got {actual}"

    print("TrOCR alignment check passed.")
    print(f"decoder_start_token_id = {expected_decoder_start}")
    print(f"pad_token_id           = {expected_pad}")
    print(f"eos_token_id           = {expected_eos}")

check_trocr_alignment(model, processor)

TrOCR alignment check passed.
decoder_start_token_id = 0
pad_token_id           = 1
eos_token_id           = 2


### Evaluation Functions

In [7]:
import evaluate

cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")


def decode_labels(labels, processor):
    labels = labels.clone()
    labels[labels == -100] = processor.tokenizer.pad_token_id

    return processor.batch_decode(
        labels,
        skip_special_tokens=True
    )


def evaluate_ocr(model, processor, val_loader, device):
    model.eval()

    total_val_loss = 0.0
    all_preds = []
    all_trues = []

    with torch.no_grad():
        for batch in val_loader:
            batch = {
                "pixel_values": batch["pixel_values"].to(device),
                "labels": batch["labels"].to(device),
            }

            outputs = model(**batch)
            loss = outputs.loss
            total_val_loss += loss.item()

            generated_ids = model.generate(
                batch["pixel_values"]
            )

            pred_texts = processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )

            true_texts = decode_labels(
                batch["labels"].cpu(),
                processor
            )

            all_preds.extend(pred_texts)
            all_trues.extend(true_texts)

    avg_val_loss = total_val_loss / len(val_loader)

    cer = cer_metric.compute(
        predictions=all_preds,
        references=all_trues
    )

    wer = wer_metric.compute(
        predictions=all_preds,
        references=all_trues
    )

    seq_acc = sum(
        pred == true for pred, true in zip(all_preds, all_trues)
    ) / len(all_trues)

    return {
        "val_loss": avg_val_loss,
        "cer": cer,
        "wer": wer,
        "seq_acc": seq_acc,
        "preds": all_preds,
        "trues": all_trues,
    }

### Testing a few batches

In [10]:
import os

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 3

output_dir = "outputs/trocr-rxhand-best"
os.makedirs(output_dir, exist_ok=True)

best_wer = float("inf")

for epoch in range(num_epochs):
    # --------------------
    # Train
    # --------------------
    model.train()

    total_train_loss = 0.0

    for step, batch in enumerate(train_loader):
        batch = {
            "pixel_values": batch["pixel_values"].to(device),
            "labels": batch["labels"].to(device),
        }

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_train_loss += loss.item()

        if step % 25 == 0:
            print(f"epoch {epoch+1}, step {step}, train loss = {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)

    val_results = evaluate_ocr(
        model=model,
        processor=processor,
        val_loader=val_loader,
        device=device
    )

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"train loss = {avg_train_loss:.4f} | "
        f"val loss = {val_results['val_loss']:.4f} | "
        f"CER = {val_results['cer']:.4f} | "
        f"WER = {val_results['wer']:.4f} | "
        f"Seq Acc = {val_results['seq_acc']:.4f}"
    )

    if val_results["wer"] < best_wer:
        best_wer = val_results["wer"]

        model.save_pretrained(output_dir)
        processor.save_pretrained(output_dir)

        print(f"Saved new best model with WER = {best_wer:.4f}")

epoch 1, step 0, train loss = 0.4318
epoch 1, step 25, train loss = 0.3426
epoch 1, step 50, train loss = 0.8657
epoch 1, step 75, train loss = 1.1653
epoch 1, step 100, train loss = 1.5458
epoch 1, step 125, train loss = 1.1551
epoch 1, step 150, train loss = 0.4431
epoch 1, step 175, train loss = 0.5436
epoch 1, step 200, train loss = 0.5805
epoch 1, step 225, train loss = 1.1418
epoch 1, step 250, train loss = 0.2836
epoch 1, step 275, train loss = 0.6465
epoch 1, step 300, train loss = 0.9839
epoch 1, step 325, train loss = 1.0424
epoch 1, step 350, train loss = 0.3373
epoch 1, step 375, train loss = 1.4065
epoch 1, step 400, train loss = 0.6148
epoch 1, step 425, train loss = 0.8561
epoch 1, step 450, train loss = 0.7531
epoch 1, step 475, train loss = 0.8802
epoch 1, step 500, train loss = 0.7621
epoch 1, step 525, train loss = 1.1984
epoch 1, step 550, train loss = 0.5208
epoch 1, step 575, train loss = 0.6778
epoch 1, step 600, train loss = 0.5059
epoch 1, step 625, train loss 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved new best model with WER = 0.6229
epoch 2, step 0, train loss = 0.3792
epoch 2, step 25, train loss = 0.3963
epoch 2, step 50, train loss = 0.1850
epoch 2, step 75, train loss = 0.0618
epoch 2, step 100, train loss = 0.5439
epoch 2, step 125, train loss = 0.5602
epoch 2, step 150, train loss = 0.4470
epoch 2, step 175, train loss = 0.5182
epoch 2, step 200, train loss = 0.4224
epoch 2, step 225, train loss = 0.7942
epoch 2, step 250, train loss = 0.8777
epoch 2, step 275, train loss = 0.6119
epoch 2, step 300, train loss = 0.7031
epoch 2, step 325, train loss = 0.9098
epoch 2, step 350, train loss = 0.8101
epoch 2, step 375, train loss = 0.3112
epoch 2, step 400, train loss = 0.5733
epoch 2, step 425, train loss = 0.3532
epoch 2, step 450, train loss = 0.8320
epoch 2, step 475, train loss = 0.8556
epoch 2, step 500, train loss = 0.7015
epoch 2, step 525, train loss = 0.2799
epoch 2, step 550, train loss = 0.2036
epoch 2, step 575, train loss = 1.1137
epoch 2, step 600, train loss 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved new best model with WER = 0.5693


### Testing Data

In [12]:
from transformers import VisionEncoderDecoderModel, TrOCRProcessor

best_processor = TrOCRProcessor.from_pretrained(output_dir)

best_model = VisionEncoderDecoderModel.from_pretrained(output_dir).to(device)

test_dataset = RxHandBDDataset(
    csv_path="/workspace/rxhandbd/RxHandBD-ML/Test_Label.csv",
    image_dir="/workspace/rxhandbd/RxHandBD-ML/Test_Set",
    processor=processor
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

check_trocr_alignment(best_model, best_processor)


test_results = evaluate_ocr(
    model=best_model,
    processor=best_processor,
    val_loader=test_loader,
    device=device
)

print(
    f"Test loss = {test_results['val_loss']:.4f} | "
    f"Test CER = {test_results['cer']:.4f} | "
    f"Test WER = {test_results['wer']:.4f} | "
    f"Test Seq Acc = {test_results['seq_acc']:.4f}"
)

for pred, true in zip(test_results["preds"][:20], test_results["trues"][:20]):
    print("PRED:", pred)
    print("TRUE:", true)
    print()

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

TrOCR alignment check passed.
decoder_start_token_id = 0
pad_token_id           = 1
eos_token_id           = 2
Test loss = 2.4277 | Test CER = 1.1100 | Test WER = 0.8973 | Test Seq Acc = 0.2502
PRED: nexe
TRUE: Nexcital

PRED: Indever
TRUE: Inderen

PRED: Indever
TRUE: Indever

PRED: Lovita
TRUE: Losita

PRED: Rivotril
TRUE: Rivotril

PRED: Asynta
TRUE: Asynta

PRED: Napa
TRUE: Napa

PRED: Eemata
TRUE: Econate

PRED: Exepim
TRUE: Exeptim

PRED: Napa
TRUE: Napa

PRED: Kacin
TRUE: Kacin

PRED: latilax
TRUE: Lubilay

PRED: t-f
TRUE: T-Cef

PRED: Algin
TRUE: Algin

PRED: Dexter
TRUE: Dexter

PRED: Eslilex
TRUE: Escilex

PRED:  Maxyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloyloy
TRUE: Traxy 500

PRED: Setra
TRUE: Setra

PRED: Alpr DS
TRUE: HPR DS

PRED: PPlicon
TRUE: palpitation

